# climate-toolkit — Colab quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/climate-toolkit/blob/main/examples/climate_toolkit_colab.ipynb)

Location-based climate, season, climatology, hazard, and projection analysis, as a Python library.

This notebook accompanies the [Use as a package](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/use-as-a-package/) guide and demos **all seven public functions**.

**Section 2 is foundational**: most of the toolkit's data sources (AgERA5, ERA5, CHIRPS, NEX-GDDP, ...) are served through Google Earth Engine, which needs a free, one-time Google Cloud project setup. Do it once and everything in this notebook unlocks. Don't have an account yet? Leave the flag off in section 2 — sections 3–5 use NASA POWER and weather stations, which need no credentials at all.

**Contents**
1. Install
2. Set up Google Earth Engine access (foundational)
3. Fetch daily climate data → pandas DataFrame
4. Seasonal climatology & statistics (auto-detected and fixed seasons)
5. Weather stations: download & validate a grid against observations
6. Earth Engine in action: gridded sources, hazards, comparisons, projections
7. Where to go next

## 1. Install

The package is not on PyPI yet, so install straight from GitHub. Takes ~1–2 minutes on Colab.

In [ ]:
%pip install -q "git+https://github.com/CGIAR-Climate-Data-Hub/climate-toolkit.git"

In [ ]:
import climate_toolkit as ct

print(f"climate_toolkit v{ct.__version__}")
print("Public API:", [n for n in ct.__all__ if not n.startswith("__")])

## 2. Set up Google Earth Engine access (foundational)

Most gridded and projection sources (`agera_5`, `era_5`, `chirps_*`, `imerg`, `terraclimate`, `cmip_6`, `nex_gddp`, ...) route through **Google Earth Engine**. This is the one piece of setup the toolkit needs — do it once and it works everywhere (Colab, your laptop, servers). It is **free for noncommercial use** (research, academia, nonprofit) and needs no credit card:

1. Go to https://code.earthengine.google.com/register with your Google account.
2. Choose **"Register a Noncommercial or Commercial Cloud project"**, then **create a new Google Cloud project** (or reuse one).
3. Usage type: **Unpaid usage** (do *not* pick "Paid usage"). Category: e.g. **Academia & Research** or **Nonprofit**.
4. Note the **project id** you registered (looks like `my-project-123456`), paste it below, set `RUN_EARTH_ENGINE = True`, and run the cell. Colab pops up a Google sign-in.

Full walkthrough with screenshots: [Getting started → Google Earth Engine credentials](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/getting_started/#2-google-earth-engine-credentials).

> **No account yet?** Leave `RUN_EARTH_ENGINE = False` and keep going — sections 3–5 run entirely credential-free (NASA POWER + weather stations). Section 6 cells will be skipped until you complete this step.

In [ ]:
RUN_EARTH_ENGINE = False  # set True once you have registered (free for noncommercial use)
GCP_PROJECT_ID = "your-ee-project-id"  # <-- your registered Cloud project id

# Tip: instead of pasting the id here, store it once in Colab's Secrets
# (key icon in the left sidebar, name it GCP_PROJECT_ID) and use:
#   from google.colab import userdata
#   GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")

if RUN_EARTH_ENGINE:
    import os

    import ee

    ee.Authenticate()  # interactive Google sign-in (works natively in Colab)
    os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
    ee.Initialize(project=GCP_PROJECT_ID)
    print("Earth Engine ready — all sources available.")
else:
    print("Earth Engine disabled — sections 3-5 still work; section 6 will be skipped.")

## 3. Fetch daily climate data → pandas DataFrame

`nasa_power` uses plain HTTPS — no Earth Engine involved. We fetch one year of daily rainfall and temperature for Nairobi, Kenya. Swap in your own coordinates.

In [ ]:
from datetime import date

from climate_toolkit.fetch_data.source_data.sources.utils.models import ClimateVariable

LAT, LON = -1.286, 36.817  # Nairobi, Kenya — swap in your own site

VARS = [
    ClimateVariable.precipitation,
    ClimateVariable.max_temperature,
    ClimateVariable.min_temperature,
]

df = ct.fetch_climate_data(
    source="nasa_power",
    location_coord=(LAT, LON),
    variables=VARS,
    date_from=date(2020, 1, 1),
    date_to=date(2020, 12, 31),
    verbose=False,
)
print("Fetch complete!")
print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# It's a normal DataFrame — plot, resample, export as usual.
df.set_index("date")["precipitation"].plot(
    figsize=(10, 3), title="Daily precipitation, Nairobi 2020 (NASA POWER)"
);

### Monthly climatology — when does the rain fall?

Before formal season detection, it helps to *see* the annual cycle. Averaging the daily data by calendar month gives a simple climatology: total precipitation per month (which months are wet) and mean temperature. For Nairobi you can read off the bimodal East African rainfall — the "long rains" (Mar–May) and "short rains" (Oct–Dec).

In [ ]:
import pandas as pd

clim = df.copy()
clim["month"] = pd.to_datetime(clim["date"]).dt.month
monthly = clim.groupby("month").agg(
    precip_mm=("precipitation", "sum"),
    tmax_mean=("max_temperature", "mean"),
    tmin_mean=("min_temperature", "mean"),
)
n_years = pd.to_datetime(df["date"]).dt.year.nunique()
monthly["precip_mm"] = (monthly["precip_mm"] / n_years).round(1)
monthly[["tmax_mean", "tmin_mean"]] = monthly[["tmax_mean", "tmin_mean"]].round(1)

ax = monthly["precip_mm"].plot.bar(figsize=(10, 3.5), color="tab:blue", alpha=0.7,
                                   title="Monthly climatology — Nairobi (NASA POWER)")
ax.set_ylabel("precipitation (mm/month)")
ax2 = ax.twinx()
ax2.plot(range(12), monthly["tmax_mean"], color="tab:red", marker="o", label="mean Tmax")
ax2.set_ylabel("temperature (°C)")
ax2.legend(loc="upper right")
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], rotation=0)
monthly

## 4. Seasonal climatology & statistics

`analyze_climate_statistics` detects growing seasons from rainfall and returns a nested dict of per-season statistics, water balance (ET0, NDWS, WRSI), and long-term-mean (LTM) summaries.

> A real climatology needs ~20+ years; the short window here keeps the demo fast and just prints a warning.

In [ ]:
stats = ct.analyze_climate_statistics(
    location_coord=(LAT, LON),
    start_year=2015,
    end_year=2020,
    source="nasa_power",
)
print("Result blocks:", sorted(stats.keys()))

### Stable multi-year seasons with `fixed_season`

Automatic season detection can find a *different number* of seasons in different years (bimodal sites, drought years). When that happens the toolkit warns that LTM windows would blend incomparable seasons:

```
[WARN] Auto-detected season counts differ across years ... Use --fixed-season for stable multi-year seasonal LTM output.
```

The Python equivalent of the CLI's `--fixed-season` flag is the `fixed_season` parameter: a `"MM-DD:MM-DD"` onset:cessation window (up to two, comma-separated — e.g. `"03-01:05-31,10-01:12-31"` for Kenya's long + short rains). Every year is then analysed over the same calendar window, so the long-term means compare like with like.

In [ ]:
import pandas as pd

stats_fixed = ct.analyze_climate_statistics(
    location_coord=(LAT, LON),
    start_year=2015,
    end_year=2020,
    source="nasa_power",
    fixed_season="03-01:08-31",  # one stable window, every year
)

# Long-term-mean summary, one row per fixed window — a proper table:
ltm = stats_fixed["ltm_season_summary"]
print("LTM mode:", ltm["mode"])
pd.json_normalize(ltm["windows"])

## 5. Weather stations: download & validate

`download_station_data` finds the nearest GHCN-Daily (or GSOD) station and downloads its daily observations. No credentials needed. Arguments are keyword-only.

In [ ]:
station = ct.download_station_data(
    station_source="ghcn_daily",
    station_coord=(LAT, LON),
    date_from=date(2020, 1, 1),
    date_to=date(2020, 12, 31),
    max_distance_km=50.0,
    auto_select="auto-1",
)
print("Station rows:", len(station))
station.head()

### Validate a gridded product against the station

`compare_station_to_grids` checks how well a gridded dataset matches on-the-ground observations at your site — works credential-free with the `nasa_power` grid. (Add `"agera_5"` to `grid_sources` once section 2 is done.)

> We restrict `variables` to precipitation here: the station nearest Nairobi records precipitation reliably but is too sparse on temperature to pass the completeness guard. Drop the `variables` argument at sites with fuller station records.

In [ ]:
validation = ct.compare_station_to_grids(
    station_source="ghcn_daily",
    station_coord=(LAT, LON),
    date_from=date(2019, 1, 1),
    date_to=date(2020, 12, 31),
    grid_sources=["nasa_power"],
    variables=[ClimateVariable.precipitation],
    verbose=False,
)
print("Validation blocks:", sorted(validation.keys()))
validation["confidence_summary"]

## 6. Earth Engine in action

Everything below uses the Earth Engine access you set up in **section 2** (`RUN_EARTH_ENGINE = True`). If you skipped it, these cells print a reminder and do nothing.

- **6.1** Fetch gridded daily data (AgERA5)
- **6.2** Seasonal statistics for a maize system
- **6.3** Crop hazard assessment
- **6.4** Focal year vs. baseline climatology
- **6.5** Side-by-side source comparison
- **6.6** Future climate projections (NEX-GDDP, 2050)

> **Runtimes.** Earth Engine cells pull multi-year daily data; expect a few minutes each, and ~10+ minutes for the 30-year baseline in 6.4. Results are cached under `outputs/cache/`, so re-runs are fast.

### 6.1 Fetch gridded daily data (AgERA5)

Same call as section 3 — only `source` changes. `agera_5` is the recommended default gridded source (temperature, humidity, wind, solar radiation + more).

In [ ]:
if RUN_EARTH_ENGINE:
    df_ee = ct.fetch_climate_data(
        source="agera_5",
        location_coord=(LAT, LON),
        variables=VARS,
        date_from=date(2020, 1, 1),
        date_to=date(2020, 12, 31),
        verbose=False,
    )
    print("Data shape:", df_ee.shape)
    display(df_ee.head())
else:
    print("Skipped — complete the Earth Engine setup in section 2 first.")

### 6.2 Seasonal statistics for a maize system

Same as section 4, but on the AgERA5 grid and with `crop_name` so season detection and water-balance parameters are crop-aware.

In [ ]:
if RUN_EARTH_ENGINE:
    stats_ee = ct.analyze_climate_statistics(
        location_coord=(LAT, LON),
        start_year=2015,
        end_year=2020,
        source="agera_5",
        crop_name="maize",
    )
    print("Result blocks:", sorted(stats_ee.keys()))
    ltm_ee = stats_ee["ltm_season_summary"]
    display(pd.json_normalize(ltm_ee["windows"]))
else:
    print("Skipped — complete the Earth Engine setup in section 2 first.")

### 6.3 Crop hazard assessment

`evaluate_hazards` screens a growing season for heat, drought, waterlogging and other crop/livestock hazards. `source="auto"` picks the recommended precipitation + temperature sources.

In [ ]:
if RUN_EARTH_ENGINE:
    hazards = ct.evaluate_hazards(
        crop_name="Maize",
        location_coord=(LAT, LON),
        date_from="2020-03-01",
        date_to="2020-08-31",
        source="auto",
    )
    print("Hazard blocks:", sorted(hazards.keys()))
else:
    print("Skipped — complete the Earth Engine setup in section 2 first.")

### 6.4 Focal year vs. baseline climatology

How did 2023 differ from the 1991–2020 baseline at this site? **Longest cell in the notebook** (~10+ min cold: it fetches 30 years of daily data). Cached afterwards.

In [ ]:
if RUN_EARTH_ENGINE:
    diff = ct.compare_climate_periods(
        location=(LAT, LON),
        baseline_start=1991,
        baseline_end=2020,
        focal_year=2023,
        source="agera_5",
        crop_name="maize",
    )
    print("Comparison blocks:", sorted(diff.keys()))
else:
    print("Skipped — complete the Earth Engine setup in section 2 first.")

### 6.5 Side-by-side source comparison

`compare_climate_sources` fetches the same site/period from multiple datasets and writes comparison reports to `output_dir`.

In [ ]:
if RUN_EARTH_ENGINE:
    import os

    comparison = ct.compare_climate_sources(
        sources=["nasa_power", "agera_5"],
        lat=LAT,
        lon=LON,
        start="2020-01-01",
        end="2020-12-31",
        output_dir="./outputs",
    )
    print("Wrote:", sorted(os.listdir("./outputs")))
else:
    print("Skipped — complete the Earth Engine setup in section 2 first.")

### 6.6 Future climate projections (NEX-GDDP, 2050)

Downscaled CMIP6 projections via `nex_gddp` — same fetch call plus a climate `model` and emissions `scenario`.

In [ ]:
if RUN_EARTH_ENGINE:
    proj = ct.fetch_climate_data(
        source="nex_gddp",
        model="GFDL-ESM4",
        scenario="ssp245",
        location_coord=(LAT, LON),
        variables=VARS,
        date_from=date(2050, 1, 1),
        date_to=date(2050, 12, 31),
        verbose=False,
    )
    print("Data shape:", proj.shape)
    display(proj.head())
    proj.set_index("date")["max_temperature"].plot(
        figsize=(10, 3), title="Projected daily max temperature, Nairobi 2050 (GFDL-ESM4, SSP2-4.5)"
    )
else:
    print("Skipped — complete the Earth Engine setup in section 2 first.")

## 7. Where to go next

You've now touched all seven public functions. To go deeper:

- **[Use as a package](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/use-as-a-package/)** — the full guide: every function's parameters, data sources, variables, caching, recipes, troubleshooting.
- **[Getting started](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/getting_started/)** — install and Earth Engine setup in detail.
- **[API reference](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/api/)** — rendered from the docstrings; or run `help(ct.fetch_climate_data)` right here.

Issues and questions: https://github.com/CGIAR-Climate-Data-Hub/climate-toolkit/issues